# 🔍 NumPy – Indexation avancée et filtrage (03)

---

## 🎯 Objectifs
- Comprendre et utiliser l'**indexation booléenne** (masques)
- Combiner plusieurs conditions avec `&`, `|`, `~`
- Utiliser `any()` et `all()` sur un axe
- Modifier des éléments ciblés par un masque
- Utiliser `np.where()` en version 1 argument (indices) et 3 arguments (if vectorisé)
- Comprendre l'**indexation fancy** (liste d'indices)
- Appliquer tout ça sur des cas concrets : **météo**, **RH**, **ventes**, **Netflix**

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import numpy as np
print("NumPy version :", np.__version__)

---
## 📝 Partie 1 – Le masque booléen

### 🔎 Qu'est-ce qu'un masque ?

Une **condition** appliquée à un tableau NumPy retourne un tableau de **booléens** de la même shape. Ce tableau s'appelle un **masque**.

```python
temp = np.array([15, 22, 27, 31, 18])

masque = temp > 20
# → [False  True  True  True False]
#     15     22    27    31   18
```

On utilise ce masque pour **filtrer** le tableau — seuls les éléments à `True` sont retenus :

```python
temp[masque]      # → [22, 27, 31]
temp[temp > 20]   # équivalent direct (sans variable intermédiaire)
```

### 🔎 Opérateurs de comparaison

| Opérateur | Signification |
|-----------|---------------|
| `>` | strictement supérieur |
| `>=` | supérieur ou égal |
| `<` | strictement inférieur |
| `<=` | inférieur ou égal |
| `==` | égal |
| `!=` | différent |

### 🔎 `any()` et `all()` — tester le masque globalement

```python
masque = np.array([False, True, True, False])

masque.any()          # → True  (au moins un True)
masque.all()          # → False (pas tous True)

# Sur un tableau 2D : tester par ligne ou par colonne
A = np.array([[1, 30, 5],
              [8, 12, 4]])

(A > 10).any(axis=1)  # Par ligne : y a-t-il AU MOINS UNE valeur > 10 ?
# → [True, True]   (ligne 0 : 30 > 10 ✓ / ligne 1 : 12 > 10 ✓)

(A > 10).all(axis=1)  # Par ligne : sont-elles TOUTES > 10 ?
# → [False, False]  (non, 1 et 5 ne le sont pas)
```

> 💡 **Cas d'usage** : `any(axis=1)` est très utile pour trouver les lignes qui contiennent au moins une valeur anormale (seuil dépassé, valeur manquante, etc.).

In [ ]:
import numpy as np

# Météo 🌡️ : températures max sur 7 jours
temp = np.array([15, 22, 27, 31, 18, 25, 29])

# Créer et afficher le masque
masque_chaud = temp > 25
print("Températures   :", temp)
print("Masque (> 25°C):", masque_chaud)
print("Jours chauds   :", temp[masque_chaud])

print()

# any / all sur le masque
print("Au moins un jour > 25°C ?", masque_chaud.any())   # True
print("Tous les jours > 25°C ? ", masque_chaud.all())    # False
print("Nombre de jours > 25°C :", masque_chaud.sum())    # sum() compte les True

print()

# Sur un tableau 2D : météo de 5 jours × 3 villes
meteo_2d = np.array([[18, 22, 25],
                     [20, 24, 31],
                     [17, 19, 23],
                     [28, 33, 35],
                     [22, 26, 29]])

# Quels jours y a-t-il AU MOINS UNE ville à plus de 30°C ?
jours_alerte = (meteo_2d >= 30).any(axis=1)
print("Jours en alerte canicule :", jours_alerte)
print("Indices des jours en alerte :", np.where(jours_alerte)[0])

👉 **Exercice 1** : Analyse RH — salaires mensuels bruts de 8 employés.

`salaires = [2100, 3500, 1800, 4200, 2750, 3100, 1650, 5000]`

1. Créez le masque des salaires supérieurs à 3000 €
2. Affichez ces salaires
3. Combien d'employés gagnent plus de 3000 € ? (utilisez `.sum()` sur le masque)
4. Y a-t-il au moins un salaire supérieur à 4500 € ? (`.any()`)
5. Tous les salaires sont-ils supérieurs à 1500 € ? (`.all()`)

In [ ]:
import numpy as np

salaires = np.array([2100, 3500, 1800, 4200, 2750, 3100, 1650, 5000])

# TODO 1 : masque salaires > 3000
masque = ...
print("Masque :", masque)

# TODO 2 : afficher les salaires concernés
print("Salaires > 3000 € :", ...)

# TODO 3 : nombre d'employés concernés
print("Nombre d'employés > 3000 € :", ...)

# TODO 4 : au moins un > 4500 ?
print("Au moins un > 4500 € ?", ...)

# TODO 5 : tous > 1500 ?
print("Tous > 1500 € ?", ...)

<details>
<summary>💡 Correction</summary>

```python
import numpy as np

salaires = np.array([2100, 3500, 1800, 4200, 2750, 3100, 1650, 5000])

masque = salaires > 3000
print("Masque :", masque)
# [False  True False  True False  True False  True]

print("Salaires > 3000 € :", salaires[masque])     # [3500 4200 3100 5000]
print("Nombre d'employés > 3000 € :", masque.sum()) # 4
print("Au moins un > 4500 € ?", (salaires > 4500).any())  # True
print("Tous > 1500 € ?", (salaires > 1500).all())          # True
```
</details>

---
## 📝 Partie 2 – Combiner plusieurs conditions

### 🔎 Opérateurs logiques vectorisés

⚠️ **Important** : avec NumPy, on n'utilise **pas** `and`, `or`, `not` (qui sont des opérateurs Python scalaires). On utilise :

| Opérateur | Signification | Exemple |
|-----------|---------------|---------|
| `&` | ET logique | `(A > 10) & (A < 30)` |
| `\|` | OU logique | `(A < 5) \| (A > 95)` |
| `~` | NON logique (négation) | `~(A > 10)` |

> ⚠️ **Les parenthèses sont obligatoires** autour de chaque condition, car `&` et `|` ont une priorité plus haute que `>` et `<`.
>
> ```python
> # ❌ ERREUR fréquente :
> temp > 20 & temp < 30     # interprété comme temp > (20 & temp) < 30
>
> # ✅ CORRECT :
> (temp > 20) & (temp < 30)
> ```

### 🔎 Filtrer des lignes entières sur un tableau 2D

Pour filtrer des **lignes** selon une condition sur **une colonne** :

```python
# Récupérer les LIGNES entières où la colonne 0 > 50
A[A[:, 0] > 50]   # A[:, 0] = tous les éléments de la colonne 0

# ⚠️ Ne pas confondre avec :
A[:, A[0, :] > 50]  # → sélectionne des COLONNES (pas des lignes !)
```

In [ ]:
import numpy as np

temp = np.array([15, 22, 27, 31, 18, 25, 29])

# ET : températures entre 20 et 28 inclus
inter = temp[(temp >= 20) & (temp <= 28)]
print("Entre 20 et 28°C :", inter)

# OU : températures extrêmes (< 18 ou > 30)
extremes = temp[(temp < 18) | (temp > 30)]
print("Extrêmes (<18 ou >30) :", extremes)

# NON : tout sauf les températures > 25
pas_chaud = temp[~(temp > 25)]
print("Pas chauds (≤ 25°C) :", pas_chaud)

print()

# Netflix 🎬 : minutes vues — 7 jours × 3 séries
np.random.seed(1)
visionnages = np.random.randint(20, 60, (7, 3))
print("Visionnages :\n", visionnages)

# Filtrer les LIGNES (jours) où la série A (colonne 0) dépasse 50 min
jours_serie_a = visionnages[visionnages[:, 0] > 50]
print("\nJours où série A > 50 min :\n", jours_serie_a)

# Filtrer les LIGNES où la série B (colonne 1) est entre 30 et 45 min
jours_serie_b = visionnages[(visionnages[:, 1] >= 30) & (visionnages[:, 1] <= 45)]
print("Jours où série B entre 30 et 45 min :\n", jours_serie_b)

👉 **Exercice 2** : Analyse des ventes — 5 produits × 6 jours.

```python
np.random.seed(7)
ventes = np.random.randint(10, 80, (6, 5))
```

1. Affichez toutes les valeurs supérieures à 60 (masque global)
2. Affichez les **lignes** (jours) où le produit 0 (colonne 0) dépasse 50
3. Affichez les valeurs entre 30 et 50 inclus
4. Combien de valeurs sont strictement inférieures à 20 ?
5. Y a-t-il des jours où **tous** les produits dépassent 40 ? (`all(axis=1)`)

In [ ]:
import numpy as np
np.random.seed(7)
ventes = np.random.randint(10, 80, (6, 5))
print("Ventes :\n", ventes)

# TODO 1 : valeurs > 60
print("\nValeurs > 60 :", ...)

# TODO 2 : lignes où produit 0 > 50
print("Jours où produit 0 > 50 :\n", ...)

# TODO 3 : valeurs entre 30 et 50 inclus
print("Valeurs entre 30 et 50 :", ...)

# TODO 4 : nombre de valeurs < 20
print("Nombre de valeurs < 20 :", ...)

# TODO 5 : jours où tous les produits > 40
print("Jours où tous > 40 :", ...)

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
np.random.seed(7)
ventes = np.random.randint(10, 80, (6, 5))
print("Ventes :\n", ventes)

# 1
print("\nValeurs > 60 :", ventes[ventes > 60])

# 2 — filtrer les LIGNES entières où colonne 0 > 50
print("Jours où produit 0 > 50 :\n", ventes[ventes[:, 0] > 50])

# 3
print("Valeurs entre 30 et 50 :", ventes[(ventes >= 30) & (ventes <= 50)])

# 4 — sum() compte les True
print("Nombre de valeurs < 20 :", (ventes < 20).sum())

# 5 — all(axis=1) : True si tous les éléments de la ligne sont > 40
print("Jours où tous > 40 :", (ventes > 40).all(axis=1))
```
</details>

---
## 📝 Partie 3 – Modifier des éléments par masque

### 🔎 Affectation conditionnelle — modifier directement via un masque

On peut utiliser un masque pour **modifier** des éléments du tableau directement. C'est une des opérations les plus pratiques de NumPy.

```python
temp = np.array([15, 22, 27, 35, 18])

# Remplacer les températures > 30 par 30 (écrêtage du capteur)
temp[temp > 30] = 30
# → [15, 22, 27, 30, 18]   le 35 est remplacé par 30

# Appliquer une remise de 10% uniquement sur les ventes > 50
ventes[ventes > 50] = ventes[ventes > 50] * 0.90
# → seules les ventes au-dessus de 50 sont réduites
```

### 🔎 `np.where` — deux usages très différents

`np.where` est une fonction qui existe en **deux formes** avec des comportements complètement différents.

---

**Forme 1 — `np.where(condition)` : "où est-ce que c'est vrai ?"**

Avec un seul argument, `np.where` retourne les **indices** des positions où la condition est vraie.

```python
temp = np.array([15, 22, 27, 35, 18])
result = np.where(temp > 25)
print(result)       # → (array([2, 3]),)   ← un tuple contenant un tableau
print(result[0])    # → [2 3]              ← les indices qui nous intéressent
```

Pourquoi le résultat est-il enveloppé dans un tuple `(...,)` ?
NumPy prévoit le cas des tableaux 3D ou plus, où il faudrait retourner les indices pour chaque dimension. Pour un tableau 1D, on a un tuple avec un seul élément → on prend `result[0]` pour obtenir directement les indices.

```python
# Usage typique : trouver les indices puis accéder aux valeurs
indices = np.where(temp > 25)[0]    # → [2 3]
print(temp[indices])                # → [27 35]  les valeurs à ces positions
```

---

**Forme 2 — `np.where(condition, valeur_si_vrai, valeur_si_faux)` : "if vectorisé"**

Avec trois arguments, `np.where` crée un **nouveau tableau** où chaque élément est remplacé selon la condition — sans modifier le tableau original.

```python
temp = np.array([15, 22, 27, 35, 18])

# Pour chaque élément : si > 25 → "Chaud", sinon → "Normal"
etat = np.where(temp > 25, "Chaud", "Normal")
# → ['Normal' 'Normal' 'Chaud' 'Chaud' 'Normal']

# Avec des valeurs numériques : bonus de 5€ si vente >= 25, sinon 0
bonus = np.where(temp >= 25, 5.0, 0.0)
# → [0. 0. 5. 5. 0.]
```

> 💡 **Différence clé entre les deux formes :**
> - `A[masque] = valeur` → **modifie A en place** (l'original change)
> - `np.where(cond, v1, v2)` → **crée un nouveau tableau** (l'original est intact)

In [ ]:
import numpy as np

# Ventes 🍕 : 7 jours
ventes = np.array([12, 22, 35, 15, 40, 18, 27], dtype=float)
print("Ventes originales :", ventes)

# --- Affectation conditionnelle ---
# Écrêter : les jours > 30, on plafonne à 30
ventes_plaf = ventes.copy()             # copie pour ne pas modifier l'original
ventes_plaf[ventes_plaf > 30] = 30
print("Après plafonnement à 30 :", ventes_plaf)

# Appliquer une remise de 20% sur les journées > 25
ventes_remise = ventes.copy()
masque_fort = ventes_remise > 25
ventes_remise[masque_fort] = ventes_remise[masque_fort] * 0.80
print("Après remise 20% sur > 25  :", ventes_remise)

print()

# --- np.where forme 1 : indices ---
indices_forts = np.where(ventes > 25)
print("Indices des jours > 25 :", indices_forts[0])
print("Valeurs correspondantes :", ventes[indices_forts])

print()

# --- np.where forme 2 : if vectorisé ---
categorie = np.where(ventes >= 25, "Forte journée", "Journée normale")
print("Catégorie par jour :", categorie)

# Avec des valeurs numériques : bonus de 5€ si ventes >= 25, sinon 0
bonus = np.where(ventes >= 25, 5.0, 0.0)
print("Bonus par jour (€) :", bonus)

### Sur un tableau 2D : météo multi-villes

In [ ]:
import numpy as np

meteo = np.array([[18, 22, 25],
                  [20, 24, 31],
                  [17, 19, 23],
                  [28, 33, 35],
                  [22, 26, 29]])

print("Météo originale :\n", meteo)

# Remplacer toutes les valeurs < 18 par 18 (température minimale capteur)
meteo_corr = meteo.copy()
meteo_corr[meteo_corr < 18] = 18
print("\nAprès correction (min 18°C) :\n", meteo_corr)

# np.where : alerte canicule si >= 30, normal sinon
alerte = np.where(meteo >= 30, "🔴 ALERTE", "🟢 Normal")
print("\nAlertes :\n", alerte)

👉 **Exercice 3** : Gestion des stocks.

Stocks de 6 produits sur 5 entrepôts (shape 5×6) :
```python
np.random.seed(3)
stocks = np.random.randint(0, 100, (5, 6))
```

1. Affichez les valeurs en **rupture** (stock = 0)
2. Comptez le nombre de cases en rupture
3. Remplacez les ruptures (= 0) par `-1` pour les signaler
4. Utilisez `np.where` pour créer un tableau de statuts : `"Rupture"` si stock original < 10, sinon `"OK"`
5. Y a-t-il des entrepôts (lignes) où **au moins un** produit est en rupture (< 10) ? Affichez leurs indices.

In [ ]:
import numpy as np
np.random.seed(3)
stocks = np.random.randint(0, 100, (5, 6))
print("Stocks :\n", stocks)

# TODO 1 : valeurs == 0 (rupture)
print("\nValeurs en rupture (=0) :", ...)

# TODO 2 : nombre de cases en rupture
print("Nombre de ruptures :", ...)

# TODO 3 : remplacer les 0 par -1 (sur une copie)
stocks_marq = stocks.copy()
stocks_marq[...] = -1
print("Stocks marqués :\n", stocks_marq)

# TODO 4 : statut par case
statut = np.where(..., "Rupture", "OK")
print("Statuts :\n", statut)

# TODO 5 : entrepôts avec au moins un produit < 10
entrepots_pb = np.where((stocks < 10).any(axis=1))[0]
print("Entrepôts avec au moins une rupture :", entrepots_pb)

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
np.random.seed(3)
stocks = np.random.randint(0, 100, (5, 6))
print("Stocks :\n", stocks)

# 1
print("\nValeurs en rupture (=0) :", stocks[stocks == 0])

# 2
print("Nombre de ruptures :", (stocks == 0).sum())

# 3
stocks_marq = stocks.copy()
stocks_marq[stocks_marq == 0] = -1
print("Stocks marqués :\n", stocks_marq)

# 4
statut = np.where(stocks < 10, "Rupture", "OK")
print("Statuts :\n", statut)

# 5 — np.where(masque_1d)[0] retourne les indices des True
entrepots_pb = np.where((stocks < 10).any(axis=1))[0]
print("Entrepôts avec au moins une rupture :", entrepots_pb)
```
</details>

---
## 📝 Partie 4 – Indexation fancy (liste d'indices)

### 🔎 Qu'est-ce que l'indexation fancy ?

Le **slicing** extrait des éléments **contigus** (une plage continue : du 2 au 5, de la colonne 1 à la 3…).  
L'**indexation fancy** permet de choisir des éléments à des positions **quelconques**, même non contigus.

**Exemple concret :** vous avez un catalogue de 7 produits et un client qui veut les produits 0, 3 et 5.

```python
prix = np.array([9.99, 24.90, 4.50, 149.0, 39.90, 12.00, 89.99])
#                  0      1     2      3       4      5      6

# Slicing : on ne peut prendre que des plages continues
prix[1:4]          # → [24.90, 4.50, 149.0]  (produits 1, 2, 3 — contigus)

# Indexation fancy : on passe une LISTE d'indices
prix[[0, 3, 5]]    # → [9.99, 149.0, 12.00]  (produits 0, 3 et 5 — pas contigus)
prix[[5, 0, 3]]    # → [12.00, 9.99, 149.0]  (dans l'ordre qu'on veut)
```

**Sur un tableau 2D :** on peut sélectionner des lignes entières non contiguës.

```python
notes = np.array([[15, 12, 18],   # Étudiant 0
                  [ 9, 14, 11],   # Étudiant 1
                  [17, 16, 19],   # Étudiant 2
                  [10, 13,  8]])  # Étudiant 3

# Récupérer uniquement les étudiants 0 et 2 (pas le 1 ni le 3)
notes[[0, 2]]     # → lignes 0 et 2 seulement
```

### 🔎 `np.argsort` — trier par indice

`np.argsort` ne trie pas le tableau — il retourne les **indices** qui permettraient de le trier.

```python
notes_moyennes = np.array([14.5, 11.0, 17.3, 10.5])

# np.sort : donne les valeurs triées
np.sort(notes_moyennes)             # → [10.5, 11.0, 14.5, 17.3]

# np.argsort : donne les INDICES qui trieraient le tableau
np.argsort(notes_moyennes)          # → [3, 1, 0, 2]
# Lecture : l'indice 3 (10.5) est le plus petit,
#           puis l'indice 1 (11.0), puis 0 (14.5), puis 2 (17.3)

# Cas pratique : classer les étudiants du meilleur au moins bon
classement = np.argsort(notes_moyennes)[::-1]   # [::-1] = ordre inverse = décroissant
# → [2, 0, 1, 3]  : étudiant 2 en premier, puis 0, puis 1, puis 3

# Puis on utilise l'indexation fancy pour récupérer les notes dans cet ordre
noms = ["Alice", "Bob", "Charlie", "Diana"]
for rang, idx in enumerate(classement):
    print(f"{rang+1}. {noms[idx]} : {notes_moyennes[idx]}")
```

> ⚠️ **Vue vs copie :** l'indexation fancy retourne toujours une **copie** — modifier le résultat ne modifie pas le tableau d'origine. C'est différent du slicing qui retourne une vue.

In [ ]:
import numpy as np

# Catalogue de produits (prix en €)
catalogue = np.array([9.99, 24.90, 4.50, 149.0, 39.90, 12.00, 89.99])
noms = ["Stylo", "Agenda", "Gomme", "Clavier", "Souris", "Câble", "Écran"]

# Le client veut les produits 1, 3 et 5
selection = catalogue[[1, 3, 5]]
print("Sélection client :")
for i, idx in enumerate([1, 3, 5]):
    print(f"  {noms[idx]} : {catalogue[idx]} €")
print("Total panier :", selection.sum(), "€")

print()

# Sur un tableau 2D : notes de 4 étudiants × 5 matières
np.random.seed(0)
notes = np.random.randint(8, 20, (4, 5))
print("Notes (étudiants × matières) :\n", notes)

# Sélectionner les étudiants 0 et 2 (non contigus)
etudiants_sel = notes[[0, 2]]
print("\nÉtudiants 0 et 2 :\n", etudiants_sel)

# Trier les étudiants par leur moyenne décroissante
moyennes = notes.mean(axis=1)
ordre = np.argsort(moyennes)[::-1]   # indices qui trieraient par ordre décroissant
print("\nMoyennes :", np.round(moyennes, 1))
print("Classement (meilleur → moins bon) : étudiants", ordre)
print("Notes triées par moyenne décroissante :\n", notes[ordre])

👉 **Exercice 4** : Classement de films.

Scores de 6 films notés sur 5 critères (histoire, image, son, acteurs, rythme) :
```python
np.random.seed(10)
scores = np.random.randint(1, 11, (6, 5))
films = ["Film A", "Film B", "Film C", "Film D", "Film E", "Film F"]
```

1. Affichez les scores des films 1, 3 et 5 (indexation fancy)
2. Calculez le score moyen de chaque film (`axis=1`)
3. Triez les films du meilleur au moins bon avec `np.argsort` (ordre décroissant)
4. Affichez le podium (les 3 meilleurs films) avec leurs noms et scores moyens

In [ ]:
import numpy as np
np.random.seed(10)
scores = np.random.randint(1, 11, (6, 5))
films  = ["Film A", "Film B", "Film C", "Film D", "Film E", "Film F"]
print("Scores :\n", scores)

# TODO 1 : films 1, 3, 5
print("\nFilms 1, 3, 5 :\n", scores[...])

# TODO 2 : score moyen par film
moyennes = ...
print("Moyennes :", np.round(moyennes, 1))

# TODO 3 : trier du meilleur au moins bon
# np.argsort retourne les indices qui trieraient le tableau par ordre croissant
# [::-1] pour inverser en décroissant
classement = ...
print("Classement (indices) :", classement)

# TODO 4 : podium (3 premiers)
print("\n🏆 Podium :")
for rang, idx in enumerate(classement[:3]):
    print(f"  {rang+1}. {films[idx]} — moyenne : {moyennes[idx]:.1f}/10")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
np.random.seed(10)
scores = np.random.randint(1, 11, (6, 5))
films  = ["Film A", "Film B", "Film C", "Film D", "Film E", "Film F"]
print("Scores :\n", scores)

# 1 — indexation fancy
print("\nFilms 1, 3, 5 :\n", scores[[1, 3, 5]])

# 2 — moyenne par film (axis=1 = on réduit les colonnes)
moyennes = scores.mean(axis=1)
print("Moyennes :", np.round(moyennes, 1))

# 3 — argsort croissant → [::-1] pour décroissant
classement = np.argsort(moyennes)[::-1]
print("Classement (indices) :", classement)

# 4 — podium
print("\n🏆 Podium :")
for rang, idx in enumerate(classement[:3]):
    print(f"  {rang+1}. {films[idx]} — moyenne : {moyennes[idx]:.1f}/10")
```
</details>

---
## 🧩 Exercice de synthèse – Analyse complète trafic web

Trafic journalier (visiteurs) pour 3 canaux sur 7 jours :

```python
canaux = ["Mobile", "Desktop", "Tablette"]
trafic = np.array([[1200, 800, 200],
                   [1400, 850, 220],
                   [1500, 900, 250],
                   [ 800, 400,  80],   # jour creux
                   [1700,1000, 280],
                   [1650, 980, 260],
                   [1750,1020, 300]])
```

1. Identifiez les **jours creux** : jours où le trafic total (`axis=1`) est inférieur à 1500 visiteurs
2. Pour ces jours creux, affichez le trafic détaillé par canal
3. Calculez la **moyenne par canal** sur les jours **non creux** seulement
4. Créez un tableau de statuts : `"Creux"` si trafic total < 1500, sinon `"Normal"` (`np.where`)
5. Sur les jours normaux, appliquez un **bonus de 5%** au canal Mobile (colonne 0) — sur une copie
6. Quel canal a le plus souvent dépassé sa propre moyenne journalière ? (indice : comparez chaque valeur à la moyenne de sa colonne)

In [ ]:
import numpy as np

canaux = ["Mobile", "Desktop", "Tablette"]
trafic = np.array([[1200,  800, 200],
                   [1400,  850, 220],
                   [1500,  900, 250],
                   [ 800,  400,  80],
                   [1700, 1000, 280],
                   [1650,  980, 260],
                   [1750, 1020, 300]])

# TODO 1 : masque jours creux (total < 1500)
total_jour = ...
masque_creux = ...
print("Jours creux (indices) :", np.where(masque_creux)[0])

# TODO 2 : trafic des jours creux
print("Trafic jours creux :\n", trafic[masque_creux])

# TODO 3 : moyenne par canal sur les jours NON creux
moy_normaux = ...
for i, canal in enumerate(canaux):
    print(f"  Moyenne {canal} (jours normaux) : {moy_normaux[i]:.0f}")

# TODO 4 : statuts
statuts = np.where(..., "Creux", "Normal")
print("Statuts :", statuts)

# TODO 5 : bonus 5% mobile sur jours normaux
trafic_bonus = trafic.copy()
masque_normal = ~masque_creux
trafic_bonus[masque_normal, 0] = ...
print("Mobile avec bonus :", trafic_bonus[:, 0])

# TODO 6 : canal qui dépasse le plus souvent sa moyenne
moy_canal = trafic.mean(axis=0)          # moyenne par colonne
depasse = trafic > moy_canal             # broadcasting automatique
nb_depassements = depasse.sum(axis=0)    # par canal
print("Dépassements par canal :", nb_depassements)
print("Canal le plus régulièrement au-dessus :", canaux[np.argmax(nb_depassements)])

<details>
<summary>💡 Correction</summary>

```python
import numpy as np

canaux = ["Mobile", "Desktop", "Tablette"]
trafic = np.array([[1200,  800, 200],
                   [1400,  850, 220],
                   [1500,  900, 250],
                   [ 800,  400,  80],
                   [1700, 1000, 280],
                   [1650,  980, 260],
                   [1750, 1020, 300]])

# 1
total_jour   = trafic.sum(axis=1)
masque_creux = total_jour < 1500
print("Jours creux (indices) :", np.where(masque_creux)[0])  # [3]

# 2
print("Trafic jours creux :\n", trafic[masque_creux])

# 3 — ~masque_creux = jours normaux
moy_normaux = trafic[~masque_creux].mean(axis=0)
for i, canal in enumerate(canaux):
    print(f"  Moyenne {canal} (jours normaux) : {moy_normaux[i]:.0f}")

# 4
statuts = np.where(masque_creux, "Creux", "Normal")
print("Statuts :", statuts)

# 5 — bonus 5% sur la colonne 0, lignes normales seulement
trafic_bonus = trafic.copy()
masque_normal = ~masque_creux
trafic_bonus[masque_normal, 0] = trafic_bonus[masque_normal, 0] * 1.05
print("Mobile avec bonus :", trafic_bonus[:, 0])

# 6 — broadcasting : trafic (7,3) > moy_canal (3,) → comparaison colonne par colonne
moy_canal       = trafic.mean(axis=0)
depasse         = trafic > moy_canal
nb_depassements = depasse.sum(axis=0)
print("Dépassements par canal :", nb_depassements)
print("Canal le plus régulièrement au-dessus :", canaux[np.argmax(nb_depassements)])
```
</details>

---
## ✅ Récapitulatif

| Concept | Syntaxe | Ce qu'il faut retenir |
|---------|---------|----------------------|
| **Masque booléen** | `A[A > 10]` | Tableau de True/False qui filtre les éléments |
| **Compter les True** | `masque.sum()` | `True` vaut 1, `False` vaut 0 |
| **any / all** | `.any()` / `.all()` | Au moins un True / Tous True — marche avec `axis` |
| **Filtrer des lignes 2D** | `A[A[:, 0] > 50]` | Condition sur une colonne, retourne des lignes entières |
| **ET / OU / NON** | `(cond1) & (cond2)` | Parenthèses obligatoires — pas `and`/`or`/`not` |
| **Affectation par masque** | `A[A > 0] = 0` | Modifie le tableau en place |
| **`np.where(cond)`** | 1 argument | Retourne les **indices** des True |
| **`np.where(c, v1, v2)`** | 3 arguments | If vectorisé — retourne un nouveau tableau |
| **Indexation fancy** | `A[[0, 2, 4]]` | Positions non contiguës — retourne une **copie** |
| **`np.argsort`** | `np.argsort(a)` | Indices qui trieraient `a` par ordre croissant |

---
📘 **Prochain notebook → `04` : Statistiques, agrégations et analyses**